# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row in `fact_content_daily_performance` = one pseudonymized content item,
for one pseudonymized client, on one report date. I'm developing on
`month=2026-03` (a mid-panel, fully-settled month) and will treat
`month=2026-06` (the `_sample` partition) as a sealed test month — never
used to build or tune label logic, only to check query mechanics.

I'll verify this grain claim below with a query: group by
(client, content, report_date) and confirm zero duplicate rows.

In [13]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

import duckdb
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_NAME')
login(token=HF_TOKEN)

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MID_MONTH = "2026-03"

SEALED_MONTH = "2026-06"

In [14]:
con.sql(f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) - COUNT(DISTINCT (client_key, content_key, report_date)) AS duplicate_grain_rows
FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/*.parquet')
""")

BinderException: Binder Error: Referenced column "client_key" not found in FROM clause!
Candidate bindings: "client_hash_id", "client_has_gsc", "client_has_ga4", "gsc_clicks", "report_date"

In [15]:
con.sql(f"""
DESCRIBE SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/*.parquet') LIMIT 0
""")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [16]:
con.sql(f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) - COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS duplicate_grain_rows
FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────────┐
│ total_rows │ duplicate_grain_rows │
│   int64    │        int64         │
├────────────┼──────────────────────┤
│    9841378 │                    0 │
└────────────┴──────────────────────┘

In [17]:
con.sql(f"""
SELECT
  COUNT(*) AS row_count,
  MIN(report_date) AS min_date,
  MAX(report_date) AS max_date,
  COUNT(DISTINCT report_date) AS n_days
FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/*.parquet')
""")

┌───────────┬────────────┬────────────┬────────┐
│ row_count │  min_date  │  max_date  │ n_days │
│   int64   │    date    │    date    │ int64  │
├───────────┼────────────┼────────────┼────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │     31 │
└───────────┴────────────┴────────────┴────────┘

In [18]:
con.sql(f"""
SELECT
  COUNT(*) AS all_rows,
  COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
  ROUND(COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) * 100.0 / COUNT(*), 1) AS pct_available
FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┬────────────────────┬───────────────┐
│ all_rows │ gsc_available_rows │ pct_available │
│  int64   │       int64        │    double     │
├──────────┼────────────────────┼───────────────┤
│  9841378 │            3611061 │          36.7 │
└──────────┴────────────────────┴───────────────┘

In [19]:
df = con.sql(f"""
SELECT
  content_hash_id,
  client_hash_id,
  report_date,
  gsc_clicks,
  gsc_impressions,
  gsc_sum_position * 1.0 / NULLIF(gsc_impressions, 0) AS avg_position,
  gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0) AS ctr
FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/*.parquet')
WHERE gsc_data_available IS TRUE
""").df()

df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,client_hash_id,report_date,gsc_clicks,gsc_impressions,avg_position,ctr
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,2026-03-01,0,20,3.350000,0.000
1,content_05597932fe4da067,client_73cda7b4e4f265ea,2026-03-01,0,1,0.000000,0.000
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,2026-03-01,1,125,4.928000,0.008
3,content_905aa32a0230694e,client_73cda7b4e4f265ea,2026-03-01,0,7,4.000000,0.000
4,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,2026-03-01,0,11,2.272727,0.000


Five features and why each is knowable at the decision moment:

1. **gsc_clicks (trailing day)** — an already-logged Search Console outcome
   for a past day; not a future value.
2. **gsc_impressions (trailing day)** — same: historical GSC data already
   synced by the decision date.
3. **avg_position = gsc_sum_position / gsc_impressions (trailing day)** —
   derived from two same-day historical fields; the ranking position on a
   past date is recorded fact.
4. **ctr = gsc_clicks / gsc_impressions (trailing day)** — a same-day ratio
   of two already-known features; no future information enters it.
5. **gsc_data_available (same-day flag)** — tells us whether this row's
   GSC fields can be trusted, known the moment the row exists; used to
   filter, not just as a feature.

In [20]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

pivot = df.groupby('content_hash_id').agg(
    early_clicks=('gsc_clicks', lambda s: s.iloc[:7].sum()),
    late_clicks=('gsc_clicks', lambda s: s.iloc[-7:].sum()),
    avg_ctr=('ctr', 'mean'),
    avg_position=('avg_position', 'mean'),
).reset_index()
pivot['needs_refresh'] = (pivot['late_clicks'] < pivot['early_clicks'] * 0.7).astype(int)

X_honest = pivot[['avg_ctr', 'avg_position']].fillna(0)
y = pivot['needs_refresh']

model = LogisticRegression(max_iter=1000).fit(X_honest, y)
honest_auc = roc_auc_score(y, model.predict_proba(X_honest)[:, 1])
print("Honest AUC:", honest_auc)

# --- THE TRAP ---
pivot['leaky_late_clicks'] = pivot['late_clicks']
X_leaky = pivot[['avg_ctr', 'avg_position', 'leaky_late_clicks']].fillna(0)
model_leaky = LogisticRegression(max_iter=1000).fit(X_leaky, y)
leaky_auc = roc_auc_score(y, model_leaky.predict_proba(X_leaky)[:, 1])
print("Leaky AUC (should jump toward 1.0):", leaky_auc)

pivot = pivot.drop(columns=['leaky_late_clicks'])
print("Final honest AUC kept:", honest_auc)

Honest AUC: 0.5835920458516607
Leaky AUC (should jump toward 1.0): 0.5836420788184307
Final honest AUC kept: 0.5835920458516607


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features** (inputs the model would see, all knowable at decision time):
- `gsc_clicks` (trailing day) — past clicks, already synced.
- `gsc_impressions` (trailing day) — past impressions, already synced.
- `avg_position` = `gsc_sum_position / gsc_impressions` (trailing day) — past
  average ranking position, derived from same-day historical fields.
- `ctr` = `gsc_clicks / gsc_impressions` (trailing day) — same-day ratio of
  two already-known fields.

**Label / proxy**:
- `needs_refresh` — a proxy built from a forward decline in `gsc_clicks`
  (last 7 days vs. first 7 days of the trailing window) relative to a
  content item's own trailing baseline. This is a proxy, not a ground-truth
  label — FlyRank doesn't hand us a "this page was refreshed and it worked"
  flag, so I'm approximating "needs attention" from an observed decline.

**Context** (used to filter/scope, not fed to the model as signal):
- `report_date` — defines the time window (`month=2026-03`), not a feature.
- `client_hash_id`, `content_hash_id` — identify the grain / group-by keys,
  not predictive signal on their own (they're pseudonymized hashes with no
  inherent meaning).
- `gsc_data_available` — used as a row filter (`WHERE ... IS TRUE`) to keep
  only reliable rows, not passed into the model.

**Excluded**:
- `late_clicks` (the last-7-days sum used to define the label) — excluded
  because it *is* the label, derived directly from the future outcome
  relative to the decision point. Including it is the leakage trap
  demonstrated later in this notebook.
- `client_has_gsc` / `client_has_ga4` (client-level, not row-level flags) —
  excluded because they describe the client's overall data coverage, not
  this specific page's performance; mixing them in would blur the grain.
- `ga4_data_available`, GA4/session/AI-referral fields (`sessions_ai`,
  `ai_chatgpt`, etc.) — excluded for this lane; content refresh here is
  scoped to Search Console signal only, to keep the slice small and
  the contract honest about what it actually covers.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Claim: one row = one (client, content item, report date).**
Verified by the grain-check query below: `duplicate_grain_rows` must be 0.

In [21]:
con.sql(f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) - COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS duplicate_grain_rows
FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────────┐
│ total_rows │ duplicate_grain_rows │
│   int64    │        int64         │
├────────────┼──────────────────────┤
│    9841378 │                    0 │
└────────────┴──────────────────────┘

**Claim: my slice is `month=2026-03`, with a row count and date span I can state exactly.**
Verified below — row count, min/max date, and number of distinct days.

In [22]:
con.sql(f"""
SELECT
  COUNT(*) AS row_count,
  MIN(report_date) AS min_date,
  MAX(report_date) AS max_date,
  COUNT(DISTINCT report_date) AS n_days
FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/*.parquet')
""")

┌───────────┬────────────┬────────────┬────────┐
│ row_count │  min_date  │  max_date  │ n_days │
│   int64   │    date    │    date    │ int64  │
├───────────┼────────────┼────────────┼────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │     31 │
└───────────┴────────────┴────────────┴────────┘

**Claim: not all rows have usable GSC data — availability must be checked, not assumed.**
Verified below: filtering with `IS TRUE` (not just truthy/non-null) shows exactly how many rows survive.

In [23]:
con.sql(f"""
SELECT
  COUNT(*) AS all_rows,
  COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
  ROUND(COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) * 100.0 / COUNT(*), 1) AS pct_available
FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/*.parquet')
""")

┌──────────┬────────────────────┬───────────────┐
│ all_rows │ gsc_available_rows │ pct_available │
│  int64   │       int64        │    double     │
├──────────┼────────────────────┼───────────────┤
│  9841378 │            3611061 │          36.7 │
└──────────┴────────────────────┴───────────────┘

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**What this data can never tell you:**

1. **Unbalanced panel depth.** Per the dataset card, `dim_clients.gsc_data_start`
   / `ga4_data_start` differ by client — some clients have much longer history
   than others as of `month=2026-03`. Any trailing-window feature (like my
   7-day early/late click comparison) is less reliable for a content item
   whose client only recently started being tracked, since "trailing 7 days"
   might not exist or might be thin. This data can't tell me whether a low
   early-window count reflects genuinely low performance or just a short
   history.

2. **GSC-only rows, no GA4.** `client_has_gsc` and `client_has_ga4` are
   independent flags — a client can have GSC coverage without GA4, so
   session-level or on-site engagement signal (`sessions_ai`, `scroll_events`,
   etc.) is simply absent for a meaningful share of rows. My lane only uses
   GSC fields for this reason, but that means I can never see whether a
   click actually led to engagement — only that it happened.

3. **No causal refresh signal.** The warehouse records performance, not
   interventions — there's no column saying "this page was refreshed on
   date X." My `needs_refresh` label is a proxy inferred from a click
   decline, not an observed outcome of an actual refresh. This data can
   never confirm whether refreshing a page actually worked; it can only
   flag candidates.

4. **Window overlap / correlated observations.** Within one month, the same
   content item appears in multiple rows (once per day), so trailing-window
   features computed within `month=2026-03` share underlying days — they
   aren't independent samples. A model trained naively on daily rows within
   one client/content pair risks treating correlated days as independent
   evidence.

5. **Pseudonymized, salted keys.** `client_hash_id` and `content_hash_id`
   carry no semantic meaning and can't be joined back to real domains,
   URLs, or query text — so this data can never support content-specific
   qualitative judgment (e.g. "this page is thin content") without a
   separate joinable field I don't have in this lane.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.